In [1]:


# # 📘 Scenario: Corporate Expansion Analysis
# # Context:
# # You are part of a business intelligence team at a multinational company. Your manager has asked you to analyze press
# releases and news articles to identify key entities such as organizations, people, locations, dates, and numbers. This
#  information will help the company track competitors’ moves and plan strategy.
# # Task:
# # You receive the following press release excerpt:
# # "Apple Inc. plans a new office in Hyderabad, India. Tim Cook announced this in March 2023. The site will create 5,000 jobs
# and focus on innovative technologies in supply chain analytics."

# # Your job is to run Named Entity Recognition (NER) using spaCy to automatically extract structured insights.


import spacy

# Load the small English model
nlp = spacy.load("en_core_web_sm")

text = """
Apple Inc. plans a new office in Hyderabad, India. Tim Cook announced this in March 2023.
The site will create 5,000 jobs and focus on innovative technologies in supply chain analytics.
"""

# Load the spaCy NLP pipeline (make sure the model is installed)
# Example: python -m spacy download en_core_web_sm
import spacy

nlp = spacy.load("en_core_web_sm")

# Example text for entity extraction
text = """
Apple Inc. was founded by Steve Jobs and Steve Wozniak in California.
It is headquartered in Cupertino and has offices in New York and London.
In 2023, Apple reported revenue of $394 billion.
"""

# Process the text with spaCy NLP pipeline
# This performs tokenization, POS tagging, dependency parsing, and NER
doc = nlp(text)

# -------------------------------------------------------------
# Print all named entities detected in the text
# Each entity will show:
# 1. The actual entity text
# 2. The entity label (type of entity)
# -------------------------------------------------------------
print("Named Entities (text, label):")

for ent in doc.ents:
    # ent.text  -> the detected entity
    # ent.label_ -> the category of entity (ORG, PERSON, GPE etc.)
    print(f"{ent.text:25} -> {ent.label_}")

# -------------------------------------------------------------
# Optional Section: Group entities by their label
# This helps summarize entities under each category
# Example:
# PERSON: Steve Jobs, Steve Wozniak
# ORG: Apple
# GPE: California, New York
# -------------------------------------------------------------

from collections import defaultdict

# defaultdict automatically creates empty lists for new keys
by_label = defaultdict(list)

# Loop through entities and group them by label
for ent in doc.ents:
    by_label[ent.label_].append(ent.text)

# -------------------------------------------------------------
# Print grouped entities
# sorted(set(items)) removes duplicates and sorts results
# -------------------------------------------------------------
print("\nEntities grouped by label:")

for label, items in by_label.items():
    print(f"{label}: {sorted(set(items))}")

Named Entities (text, label):
Apple Inc.                -> ORG
Steve Jobs                -> PERSON
Steve Wozniak             -> PERSON
California                -> GPE
Cupertino                 -> GPE
New York                  -> GPE
London                    -> GPE
2023                      -> DATE
Apple                     -> ORG
$394 billion              -> MONEY

Entities grouped by label:
ORG: ['Apple', 'Apple Inc.']
PERSON: ['Steve Jobs', 'Steve Wozniak']
GPE: ['California', 'Cupertino', 'London', 'New York']
DATE: ['2023']
MONEY: ['$394 billion']


In [2]:
# Word embeddings with gensim Word2Vec
# Setup: python -m pip install gensim nltk

import nltk
nltk.download("punkt")
nltk.download("punkt_tab") # Added to resolve LookupError
from nltk.tokenize import sent_tokenize, word_tokenize

from gensim.models import Word2Vec

corpus = """
Manufacturing relies on predictive maintenance and supply chain optimization.
Data engineers build pipelines, while analysts monitor KPIs and anomalies.
Robotics and IoT sensors stream telemetry to cloud databases for real-time insights.
Quality control uses computer vision to detect defects on the shop floor.
"""

# Tokenize sentences -> words
sentences = [word_tokenize(s.lower()) for s in sent_tokenize(corpus)]

# Train a small Word2Vec model
model = Word2Vec(
    sentences,
    vector_size=50,   # embedding dimension
    window=5,         # context window size
    min_count=1,      # keep all words for demo
    workers=2,
    sg=1              # skip-gram; use 0 for CBOW
)

# Explore similar words
for target in ["maintenance", "supply", "quality", "telemetry"]:
    print(f"\nTop similar to '{target}':")
    try:
        for w, score in model.wv.most_similar(target, topn=5):
            print(f"{w:15} -> {score:.3f}")
    except KeyError:
        print("Word not in vocabulary.")

# Cosine similarity between pairs
from numpy import dot
from numpy.linalg import norm

def cosine(u, v):
    return dot(u, v) / (norm(u) * norm(v))

pairs = [("maintenance", "telemetry"),
         ("quality", "defects"),
         ("supply", "optimization")]

print("\nCosine similarities:")
for a, b in pairs:
    try:
        sim = cosine(model.wv[a], model.wv[b])
        print(f"{a:12} ~ {b:12} -> {sim:.3f}")
    except KeyError:
        print(f"Missing word: {a} or {b}")

[nltk_data] Downloading package punkt to C:\Users\ANKIT
[nltk_data]     RAJ\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\ANKIT
[nltk_data]     RAJ\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!



Top similar to 'maintenance':
real-time       -> 0.240
defects         -> 0.240
analysts        -> 0.236
kpis            -> 0.224
detect          -> 0.204

Top similar to 'supply':
optimization    -> 0.286
to              -> 0.252
cloud           -> 0.192
analysts        -> 0.169
maintenance     -> 0.169

Top similar to 'quality':
shop            -> 0.307
cloud           -> 0.267
uses            -> 0.191
floor           -> 0.154
.               -> 0.151

Top similar to 'telemetry':
predictive      -> 0.171
cloud           -> 0.171
manufacturing   -> 0.167
iot             -> 0.166
for             -> 0.164

Cosine similarities:
maintenance  ~ telemetry    -> -0.145
quality      ~ defects      -> 0.119
supply       ~ optimization -> 0.286


In [3]:
!pip install gensim

Defaulting to user installation because normal site-packages is not writeable


In [4]:
# Code Demo (Trigram counts)
from collections import defaultdict

corpus = "the cat sat on the mat the cat lay on the rug the dog barked loudly"
tokens = corpus.split()

# Build trigram counts
trigram_counts = defaultdict(lambda: defaultdict(int))
for i in range(len(tokens)-2):
    context = (tokens[i], tokens[i+1])
    next_word = tokens[i+2]
    trigram_counts[context][next_word] += 1

# Predict next word after "the cat"
context = ("the", "cat")
print("Next word predictions for context:", context)
for word, count in trigram_counts[context].items():
    print(f"{word} -> {count}")

Next word predictions for context: ('the', 'cat')
sat -> 1
lay -> 1


In [5]:
# Code Demo (Simple feedforward LM with Keras)
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, Flatten

# Vocabulary size and embedding dimension
vocab_size = 50
embed_dim = 8

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embed_dim, input_length=2), # context length=2
    Flatten(),
    Dense(32, activation='relu'),
    Dense(vocab_size, activation='softmax')  # predict next word
])

model.compile(optimizer='adam', loss='categorical_crossentropy')
model.build(input_shape=(None, 2))  # batch size flexible, sequence length = 2

model.summary()

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
#Code Demo (LSTM for next‑word prediction)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
import numpy as np

# Vocabulary size and embedding dimension
vocab_size = 100
embed_dim = 16
seq_length = 5  # input sequence length

# Define the model
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embed_dim, input_length=seq_length),
    LSTM(64),  # LSTM layer with 64 units
    Dense(vocab_size, activation='softmax')  # predict next word
])

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy')

# Force the model to build by providing input shape
model.build(input_shape=(None, seq_length))

# Show the summary
model.summary()

# Optional: run a dummy prediction to confirm
dummy_input = np.random.randint(0, vocab_size, (1, seq_length))
print("Dummy prediction shape:", model.predict(dummy_input).shape)

In [ ]:
from transformers import pipeline

# Load the sentiment analysis pipeline
classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# Input text
text = "I absolutely love this movie! Great acting."

# Perform classification
result = classifier(text)

# Display result
print(result)

In [ ]:
from keras.models import Sequential
from keras.layers import Embedding, LSTM, Dense
import numpy as np # Import numpy for dummy data generation

# Define vocabulary size and sequence length
vocab_size = 100
seq_len = 5

# Define model
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128), # Removed input_length as it's deprecated
    LSTM(256, return_sequences=True),
    LSTM(256),
    Dense(vocab_size, activation='softmax')
])

# Compile model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',  # use categorical_crossentropy if y_train is one-hot
    metrics=['accuracy']
)

# Generate dummy data for demonstration
num_samples = 100 # Number of dummy samples
X_train = np.random.randint(0, vocab_size, (num_samples, seq_len))
y_train = np.random.randint(0, vocab_size, (num_samples,))

# Train model
history = model.fit(
    X_train,
    y_train,
    epochs=2, # Reduced epochs for a quick demo run
    batch_size=64,          # adjust based on dataset size
    validation_split=0.2,   # or use validation_data=(X_val, y_val)
    verbose=1
)

Scenario: Sports Event Analysis

Context:

You are part of a sports analytics team working for a global media company. Your manager has asked you to analyze match reports and press releases to identify key entities such as teams, players, locations, dates, and numbers. This information will help the company build automated summaries and highlight reels for fans.

Task:

You receive the following match report excerpt:

"Manchester United defeated Real Madrid 3-2 at Wembley Stadium, London. Cristiano Ronaldo scored twice, while Marcus Rashford netted the winning goal in July 2024"

Your job is to run Named Entity Recognition (NER) using spacy to automatically extract structured insights.

14:44

Type a message

In [ ]:
!pip install spacy
!python -m spacy download en_core_web_sm

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")

text = """
Manchester United defeated Real Madrid 3-2 at Wembley Stadium, London.
Cristiano Ronaldo scored twice, while Marcus Rashford netted the winning goal in July 2024.
"""

doc = nlp(text)

print("Extracted Named Entities:\n")

for ent in doc.ents:
    print(ent.text, "->", ent.label_)

📘 Scenario: Sports Event Analysis
Context:
You are part of a sports analytics team working for a global media company. Your manager has asked you to analyze match reports and press releases to identify key entities such as teams, players, locations, dates, and numbers. This information will help the company build automated summaries and highlight reels for fans.
Task:
You receive the following match report excerpt:
"Manchester United defeated Real Madrid 3-2 at Wembley Stadium, London. Cristiano Ronaldo scored twice, while Marcus Rashford netted the winning goal in July 2024."

Your job is to run Named Entity Recognition (NER) using spaCy to automatically extract structured insights.

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")

text = "Manchester United defeated Real Madrid 3-2 at Wembley Stadium, London.Cristiano Ronaldo scored twice, while Marcus Rashford netted the winning goal in July 2024."

doc = nlp(text)

print("Extracted Named Entities:\n")

for ent in doc.ents:
    print(ent.text, "->", ent.label_)

# 📘 Scenario: Smart Manufacturing Insights
# Context:
# You are part of a data science team in a smart manufacturing company. Your manager wants you to explore how different 
# technical concepts (like maintenance, telemetry, quality control, supply chain) are related in company documents. 
# By training a small Word2Vec model, you can uncover semantic similarities between words and measure how closely they are
#  connected. This helps in building recommendation systems, knowledge graphs, or even automated reporting tools.
# Task:
# Given a short corpus describing manufacturing operations, train a Word2Vec model and analyze word similarities.


# Word embeddings with gensim Word2Vec
# Setup: python -m pip install gensim nltk

In [ ]:
import nltk
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')

corpus = [
    "Predictive maintenance reduces machine downtime",
    "Sensors collect telemetry data from machines",
    "Quality control ensures product reliability",
    "Supply chain optimization improves production efficiency",
    "Machine learning helps predictive maintenance systems",
    "Telemetry data helps monitor equipment health",
    "Quality inspection detects manufacturing defects",
    "Supply chain delays affect production planning",
    "Maintenance teams repair faulty equipment",
    "Manufacturing analytics improves operational efficiency"
]

processed_corpus = [word_tokenize(sentence.lower()) for sentence in corpus]

print("\nTokenized Corpus:")
print(processed_corpus)

model = Word2Vec(
    sentences=processed_corpus,
    vector_size=50,
    window=3,
    min_count=1,
    workers=4
)

print("\nWords similar to 'maintenance':")
print(model.wv.most_similar("maintenance"))

similarity = model.wv.similarity("maintenance", "machine")
print("\nSimilarity between 'maintenance' and 'machine':", similarity)

odd_word = model.wv.doesnt_match(["maintenance", "machine", "telemetry", "banana"])
print("\nOdd word out:", odd_word)

print("\nWord vector for 'telemetry':")
print(model.wv["telemetry"][:10])

print("\nVocabulary:")
print(list(model.wv.index_to_key))

In [ ]:
!python -m pip install gensim nltk

In [ ]:
# Scenario-Based Assessment
# Case Study: AI Assistant for a University Helpdesk

# A large university receives thousands of student queries daily related to admissions, fees, hostel facilities, course
#  schedules, and exam results.

# Currently, these queries are answered manually by staff, causing:

# Slow response time

# High operational cost

# Inconsistent responses

# The university plans to develop an AI-powered chatbot using Generative AI and Large Language Models (LLMs) to
#  automatically respond to student questions.

# You are part of the AI consulting team tasked with designing the solution.

# ==========================================
# UNIVERSITY AI HELPDESK CHATBOT
# Demonstrates:
# Generative AI
# Large Language Models (LLMs)
# Tokenization
# Transformer Inference
# Prompt Engineering
# ==========================================

# Install required libraries first
# pip install transformers torch

In [ ]:
from transformers import pipeline
import textwrap

print("Loading the chatbot. Please wait...")

chatbot = pipeline(
    "text-generation",
    model="gpt2",
    pad_token_id=50256
)

print("Model Loaded Successfully!")

knowledge_base = {
"hostel": "Hostel admission requires an admission confirmation letter, government ID proof, and two passport size photographs.",
"registration": "Course registration happens online through the university portal before the semester deadline.",
"fees": "Fees can be paid through the student dashboard using debit card, credit card, or net banking.",
"exam": "Semester exams usually start in December and May.",
"library": "Students must carry their university ID card to access the library."
}

def search_knowledge(question):
    q = question.lower()

    for key in knowledge_base:
        if key in q:
            return knowledge_base[key]

    return "Information not available in the university database."

def build_prompt(question, context):

    prompt = f"""
University Helpdesk Assistant

Use ONLY the information below to answer the student question.

Information:
{context}

Student Question: {question}

Answer:
"""
    return prompt

def get_helpdesk_response(question):

    context = search_knowledge(question)
    prompt = build_prompt(question, context)

    response = chatbot(
        prompt,
        max_new_tokens=40,
        do_sample=False
    )

    text = response[0]["generated_text"]
    answer = text.split("Answer:")[-1].strip()

    return answer

print("\nUNIVERSITY AI HELPDESK")
print("Type 'exit' to quit")

while True:

    question = input("\nStudent: ")

    if question.lower() == "exit":
        print("Goodbye!")
        break

    answer = get_helpdesk_response(question)

    wrapped = textwrap.fill(answer, width=60)
    print("\nBot:", wrapped)

Loading the chatbot. Please wait...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model Loaded Successfully!

UNIVERSITY AI HELPDESK
Type 'exit' to quit



Student:  what do i need t0 access library


Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Bot: Student Question: how can i pay



Student:  how fee can be pais


Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Bot: Student Question: how fee can be pais
